# Movie Recommendation Model

In [47]:
import numpy as np
import pandas as pd
import ast
import difflib
import pickle
from pathlib import Path

## 1 Load dataset


In [48]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')
movies.shape, credits.shape

((4803, 20), (4803, 4))

In [49]:
movies = movies.merge(credits, on='title')
print(movies.shape)
movies.head(2)

(4809, 23)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,285,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [50]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4809 non-null   int64  
 1   genres                4809 non-null   object 
 2   homepage              1713 non-null   object 
 3   id                    4809 non-null   int64  
 4   keywords              4809 non-null   object 
 5   original_language     4809 non-null   object 
 6   original_title        4809 non-null   object 
 7   overview              4806 non-null   object 
 8   popularity            4809 non-null   float64
 9   production_companies  4809 non-null   object 
 10  production_countries  4809 non-null   object 
 11  release_date          4808 non-null   object 
 12  revenue               4809 non-null   int64  
 13  runtime               4807 non-null   float64
 14  spoken_languages      4809 non-null   object 
 15  status               

## 2 Select Columns

In [51]:
#Decide which columns required mostly
movies = movies[['movie_id','title','genres','keywords','overview','release_date','vote_average','cast','crew']]
movies.head(2)

,movie_id,title,genres,keywords,overview,release_date,vote_average,cast,crew
0,19995,Avatar,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","In the 22nd century, a paraplegic Marine is di...",2009-12-10,7.2,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","Captain Barbossa, long believed to be dead, ha...",2007-05-19,6.9,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [52]:
movies.isnull().sum()

movie_id        0
title           0
genres          0
keywords        0
overview        3
release_date    1
vote_average    0
cast            0
crew            0
dtype: int64

In [53]:
#we have 3 missing values in overview and 1 missing value in release date
#it's not a big number so we remove these rows
movies.dropna(inplace=True)

In [54]:
movies.isnull().sum()

movie_id        0
title           0
genres          0
keywords        0
overview        0
release_date    0
vote_average    0
cast            0
crew            0
dtype: int64

In [55]:
movies.duplicated().sum()

np.int64(0)

## 3 Parse the stringified JSON columns

### We can see that the dictionary in the 'genres' column is currently in string format; we need to convert it into a list.
* If you want to convert the string representation of a Python data structure (such as a dictionary) into an actual Python object, use `ast.literal_eval`. It evaluates only literals, thereby preventing the execution of malicious code.

In [56]:
import ast
ast.literal_eval('[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]')

[{'id': 28, 'name': 'Action'},
 {'id': 12, 'name': 'Adventure'},
 {'id': 14, 'name': 'Fantasy'},
 {'id': 878, 'name': 'Science Fiction'}]

In [57]:
def parse_names(raw, limit=None, job_filter=None):
    """Safely parse a stringified list of dicts like '[{"name": "Action"}, ...]'
    and return a list of the 'name' values.

    limit: only take the first N entries (used for cast)
    job_filter: only keep entries where entry['job'] == job_filter (used for crew/director)
    """
    if pd.isna(raw):
        return []
    try:
        parsed = ast.literal_eval(raw) if isinstance(raw, str) else raw
    except (ValueError, SyntaxError):
        return []
    if not isinstance(parsed, list):
        return []

    names = []
    for entry in parsed:
        if not isinstance(entry, dict) or 'name' not in entry:
            continue
        if job_filter is not None and entry.get('job') != job_filter:
            continue
        names.append(entry['name'])
        if limit is not None and len(names) >= limit:
            break
    return names

In [58]:
movies['genres'] = movies['genres'].apply(parse_names)
movies['keywords'] = movies['keywords'].apply(parse_names)
movies['cast'] = movies['cast'].apply(lambda x: parse_names(x, limit=3))
movies['crew'] = movies['crew'].apply(lambda x: parse_names(x, job_filter='Director'))
movies['overview_tokens'] = movies['overview'].apply(lambda x: x.split()) #for overview column we convert all string valu into list
movies.head(2)

,movie_id,title,genres,keywords,overview,release_date,vote_average,cast,crew,overview_tokens
0,19995,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","In the 22nd century, a paraplegic Marine is di...",2009-12-10,7.2,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","Captain Barbossa, long believed to be dead, ha...",2007-05-19,6.9,"[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski],"[Captain, Barbossa,, long, believed, to, be, d..."


In [59]:
# Remove spaces inside multi-word names so 'Science Fiction' -> 'ScienceFiction'
# (otherwise the vectorizer would treat 'Science' and 'Fiction' as separate tokens
# shared with unrelated words elsewhere)
def squash_spaces(items):
    return [i.replace(' ', '') for i in items]

for col in ['genres', 'keywords', 'cast', 'crew']:
    movies[col] = movies[col].apply(squash_spaces)

In [60]:
movies.head(3)

,movie_id,title,genres,keywords,overview,release_date,vote_average,cast,crew,overview_tokens
0,19995,Avatar,"[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","In the 22nd century, a paraplegic Marine is di...",2009-12-10,7.2,"[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","Captain Barbossa, long believed to be dead, ha...",2007-05-19,6.9,"[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski],"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...",A cryptic message from Bond’s past sends him o...,2015-10-26,6.3,"[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes],"[A, cryptic, message, from, Bond’s, past, send..."


## 4. Build tags

genres and keywords are the strongest genre-identity signal, so we repeat them to give them more weight relative to cast/crew/overview in the bag-of-words. (In the original, one shared cast member could outweigh a shared genre.)

In [61]:
GENRE_WEIGHT = 3      # repeat genres so they dominate the similarity signal
KEYWORD_WEIGHT = 2

movies['tags'] = (
    movies['genres'] * GENRE_WEIGHT
    + movies['keywords'] * KEYWORD_WEIGHT
    + movies['cast']
    + movies['crew']
    + movies['overview_tokens']
)

catalog = movies[['movie_id', 'title', 'genres', 'vote_average', 'tags']].copy()
catalog['tags'] = catalog['tags'].apply(lambda x: ' '.join(x).lower())
catalog.head(2)

,movie_id,title,genres,vote_average,tags
0,19995,Avatar,"[Action, Adventure, Fantasy, ScienceFiction]",7.2,action adventure fantasy sciencefiction action...
1,285,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]",6.9,adventure fantasy action adventure fantasy act...


## 5 The Natural Language Toolkit (NLTK)
**Stemming**

In [62]:
#!pip install nltk
#The Natural Language Toolkit (NLTK) is one of the most popular and foundational open-source Python libraries used for Natural Language Processing (NLP) and computational linguistics.

from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
#Creating a function where Stemming reduce words to their base forms
def stem(text):
    return ' '.join(ps.stem(word) for word in text.split())

catalog['tags'] = catalog['tags'].apply(stem)
catalog['tags'].iloc[0][:200]

'action adventur fantasi sciencefict action adventur fantasi sciencefict action adventur fantasi sciencefict cultureclash futur spacewar spacecoloni societi spacetravel futurist romanc space alien trib'

## 6 Vectorization

In [63]:
new_df['tags'][0]

NameError: name 'new_df' is not defined

In [ ]:
new_df['tags'][1]

In [ ]:
#Turning tags values into vector format
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(max_features=8000, stop_words='english', ngram_range=(1, 2))
vectors = tfidf.fit_transform(catalog['tags'])
similarity = cosine_similarity(vectors)
vectors.shape, similarity.shape
#Stopwords are common words (like "the", "is", "at") that often don't add significant meaning to text analysis.

In [ ]:
vectors

## 7. Recommend function

This version:
- matches case-insensitively
- suggests close matches instead of crashing when the title isn't found
- returns a DataFrame instead of just printing, so results can be reused/tested

In [ ]:
# Create recommendation function

title_lookup = {t.lower(): i for i, t in enumerate(catalog['title'])}

def recommend(movie, top_n=7):
    key = movie.strip().lower()
    if key not in title_lookup:
        close = difflib.get_close_matches(movie, catalog['title'], n=5, cutoff=0.4)
        if close:
            print(f"'{movie}' not found. Did you mean: {', '.join(close)}?")
        else:
            print(f"'{movie}' not found in the catalog.")
        return pd.DataFrame(columns=['title', 'genres', 'similarity'])

    idx = title_lookup[key]
    scores = list(enumerate(similarity[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:top_n + 1]

    return pd.DataFrame([
        {
            'title': catalog.iloc[i]['title'],
            'genres': catalog.iloc[i]['genres'],
            'similarity': round(score, 3),
        }
        for i, score in scores
    ])

In [ ]:
recommend('Batman Begins')

In [ ]:
# Robustness check - typo / wrong case should no longer crash
print(recommend('batman begins'))
print(recommend('Batmn Beginz'))

## 8. Quick quality check

This is a rough proxy, not a rigorous metric: for a sample of movies, what fraction of each movie's genres are shared, on average, with its top recommendations? Higher = recommendations are more genre-consistent with the query movie.

In [64]:
def genre_overlap_score(movie, top_n=7):
    key = movie.strip().lower()
    if key not in title_lookup:
        return None
    query_genres = set(catalog.iloc[title_lookup[key]]['genres'])
    if not query_genres:
        return None
    recs = recommend(movie, top_n=top_n)
    if recs.empty:
        return None
    overlaps = [
        len(query_genres & set(g)) / len(query_genres)
        for g in recs['genres']
    ]
    return sum(overlaps) / len(overlaps)

sample_titles = catalog['title'].sample(30, random_state=42)
scores = [s for s in (genre_overlap_score(t) for t in sample_titles) if s is not None]
print(f"Average genre-overlap score over {len(scores)} sampled movies: {np.mean(scores):.2f}")

Average genre-overlap score over 30 sampled movies: 0.92


## 9. Save artifacts

So the model can be reused (e.g. in a Streamlit app) without rerunning the whole pipeline.

In [65]:
Path('artifacts').mkdir(exist_ok=True)
pickle.dump(catalog[['movie_id', 'title', 'genres']], open('artifacts/movies.pkl', 'wb'))
pickle.dump(similarity, open('artifacts/similarity.pkl', 'wb'))
print('Saved artifacts/movies.pkl and artifacts/similarity.pkl')

Saved artifacts/movies.pkl and artifacts/similarity.pkl
